In [2]:
import pandas as pd

In [3]:
import pandas as pd

docs = [
    "I love this product, it works perfectly",
    "The service was terrible and very slow",
    "Average experience, nothing special",
    "Excellent quality and fast delivery",
    "I am disappointed with the purchase",
    "Not bad, but could be better",
    "Absolutely amazing! Highly recommend it",
    "Worst experience ever",
    "It is okay for the price",
    "Very satisfied with the customer support"
]
df_docs = pd.DataFrame({"text":docs})
df_docs.head()

,text
0,"I love this product, it works perfectly"
1,The service was terrible and very slow
2,"Average experience, nothing special"
3,Excellent quality and fast delivery
4,I am disappointed with the purchase


In [4]:
# cleaning text
import string
def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("","",string.punctuation))
    return text
df_docs["clean_text"] = df_docs["text"].apply(clean_text)

In [5]:
#generating embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

embeded_model = SentenceTransformer("all-miniLM-L6-v2")
doc_embeddigs = embeded_model.encode(df_docs["clean_text"].tolist(),normalize_embeddings=True)

d:\Machine Learing\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
#building FAISS index 
import faiss
embedding_dim = doc_embeddigs.shape[1]
index = faiss.IndexFlatIP(embedding_dim)
index.add(np.array(doc_embeddigs).astype("float32"))

In [7]:
#retrievel funtion
def retrieve(query , df_docs, embeded_model , index , top_k =3):
    query_emb = embeded_model.encode([query.lower()],normalize_embeddings=True).astype("float32")
    scores , indices = index.search(query_emb,top_k)
    retrieved_docs = df_docs.iloc[indices[0]]["text"].tolist()
    return retrieved_docs

In [8]:
from transformers import  AutoTokenizer,AutoModelForSeq2SeqLM

llm_model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(llm_model_name)

def generate_answer(query, retrieved_docs, max_new_tokens=100):
    context = "\n".join(retrieved_docs)
    prompt = f"Use the following context to answer the question:\n{context}\n\nQuestion: {query}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [ ]:
def generate_answer(query, retrieved_docs):
    context = "\n".join(retrieved_docs)

    prompt = (
        "You are a helpful assistant.\n"
        "Answer the question using ONLY the context below.\n"
        "Do NOT repeat the question.\n"
        "Do NOT write code.\n"
        "Do NOT list steps.\n"
        "Write a short, clear paragraph answer.\n\n"
        f"Context:\n{context}\n\n"
        f"Question:\n{query}\n\n"
        "Answer:"
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        temperature=0.5,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id
    )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)


    if "Answer:" in text:
        text = text.split("Answer:")[-1]

    return text.strip()


In [ ]:
query = "was the service slow?"

retrieved = retrieve(
    query,
    df_docs,
    embeded_model,
    index,
    top_k=3
)

print("Retrieved Documents:")
for doc in retrieved:
    print("-", doc)

answer = generate_answer(query, retrieved)
print("\nLLM Answer:\n", answer)


Retrieved Documents:
- The service was terrible and very slow
- Very satisfied with the customer support
- Excellent quality and fast delivery

LLM Answer:
 no
